# SwingTrade v2 — Full Pipeline Execution

**Periode data:** 2020-01-01 → 2026-04-01 (semua koin seragam)
**Koin:** 18 koin (5 training + 13 new)

Jalankan sel berurutan dari atas ke bawah.

---
## 0. Setup Environment
---

In [ ]:
# Deteksi environment: Colab, RunPod, atau Local
import os
import sys
from pathlib import Path

IN_COLAB  = 'google.colab' in sys.modules
IN_RUNPOD = os.path.exists('/workspace')
IS_LOCAL  = not (IN_COLAB or IN_RUNPOD)

if IN_COLAB or IN_RUNPOD:
    # Cloud: clone repo jika belum ada
    if not Path('Riset_pemodelan').exists():
        !git clone https://github.com/heathclif-cyber/Riset_pemodelan.git
    %cd /workspace/Riset_pemodelan
    
    # Install dependencies
    !pip install -q lightgbm torch scikit-learn shap pyarrow joblib pandas numpy matplotlib
else:
    # Local Windows: path = direktori notebook ini
    NOTEBOOK_DIR = Path.cwd()
    os.chdir(str(NOTEBOOK_DIR))
    print(f"Local mode: {NOTEBOOK_DIR}")

print(f"\nEnvironment: {'COLAB' if IN_COLAB else 'RUNPOD' if IN_RUNPOD else 'LOCAL'}")

In [ ]:
# Cek GPU
import torch
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU device: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
else:
    print("WARNING: No GPU detected. LSTM training will be very slow on CPU.")

---
## Phase 1: Fetch Data dari Binance
---

Download klines (1h, 4h, 1d), open interest, funding rate, dan macro data.

Estimasi: ~20-30 menit untuk 18 koin × 6 tahun.
Gunakan `--reset` untuk fetch ulang dari awal.

In [ ]:
!python pipeline/01_fetch.py --all --reset

In [ ]:
# Verifikasi hasil fetch
import os
raw_dir = Path('data/raw/klines')
if raw_dir.exists():
    coins = sorted(os.listdir(raw_dir))
    print(f"Koin ter-fetch: {len(coins)}")
    for c in coins:
        files = os.listdir(raw_dir / c)
        print(f"  {c}: {files}")
else:
    print("data/raw/klines/ belum ada — fetch mungkin belum selesai.")

---
## Phase 2: Clean Data
---

Validasi OHLC integrity, deteksi gap, join timeframe (1h+4h+1d), attach OI/funding/macro.

In [ ]:
!python pipeline/02_clean.py --all

---
## Phase 3: Feature Engineering & Labeling
---

Generate ~85 fitur + swing-based labeling v3 (SHORT/FLAT/LONG).

In [ ]:
!python pipeline/03_engineer.py --all

In [ ]:
# Analisis distribusi label dan min hold
!python pipeline/analyze_min_hold.py --save-plot

from IPython.display import Image
plot_path = Path('reports/min_hold_analysis.png')
if plot_path.exists():
    display(Image(str(plot_path)))

---
## Phase 4: Train LightGBM
---

Walk-forward CV dengan 8 folds, 24-bar gap, balanced class weights.

In [ ]:
!python pipeline/04_train_lgbm.py --all --run-id v3_full

---
## Phase 5: Train LSTM
---

Sequence model 32-bar lookback, 128 hidden dim, dropout 0.3.

**Gunakan GPU!** CPU akan sangat lambat untuk 18 koin.

In [ ]:
!python pipeline/05_train_lstm.py --all --run-id v3_full

---
## Phase 6: Stacking Ensemble
---

Logistic Regression meta-learner + isotonic calibration.

Membutuhkan: `lgbm_baseline.pkl`, `lstm_best.pt`, `lstm_scaler.pkl`

In [ ]:
!python pipeline/06_ensemble.py --all --run-id v3_full

---
## Create Model Registry
---

In [ ]:
import json
from pathlib import Path

registry = {
    "active": "ensemble_v2",
    "models": {
        "ensemble_v2": {
            "version": "v3",
            "status": "training_complete",
            "f1_macro": None,
            "winrate": None,
            "trade_per_month": None,
            "pnl_lev5x": None,
            "max_drawdown": None,
            "max_consecutive_loss": None,
            "trained_date": None,
        }
    }
}

path = Path("models/model_registry.json")
path.parent.mkdir(parents=True, exist_ok=True)
with open(path, "w") as f:
    json.dump(registry, f, indent=2)
print("model_registry.json created")

---
## Phase 7: Evaluate (SHAP)
---

Feature importance analysis + trading metrics.

In [ ]:
!python pipeline/07_evaluate.py --run-id v3_full

---
## Phase 8: Walk-Forward Backtest
---

Simulasi trading dengan ATR-based TP/SL, confidence thresholds.
Menghasilkan `inference_config.json` untuk deployment.

In [ ]:
!python pipeline/08_backtest.py --all --run-id v3_full

---
## Phase 9: Hold-Out Backtest (OOS Validation)
---

Genuine out-of-sample validation. Fetch data baru, inference WITHOUT retraining.
Default periode: 2025-05-01 → 2026-04-01.

In [ ]:
!python pipeline/09_holdout_backtest.py --all --run-id v3_full

---
## Phase 10: Visualize
---

Generate candlestick + trade charts.

In [ ]:
!python pipeline/10_visualize.py --all --holdout --verify-swing

---
## Selesai!
---

Semua hasil tersimpan di:
- **Model artifacts:** `models/lgbm_baseline.pkl`, `models/lstm_best.pt`, `models/ensemble_meta.pkl`, `models/calibrator.pkl`
- **Run artifacts:** `models/runs/v3_full/`
- **Inference config:** `models/runs/v3_full/inference_config.json`
- **Charts:** `models/runs/v3_full/charts/`
- **Reports:** `reports/`